# Kids Learning App - Database Explorer

In [3]:
import duckdb
import json
import os

def load_all_tables(family_id, kid_id=None, root_dir=None):
    """Load all tables from a kid's DB.
    Returns (table_names, dfs) where dfs[i] is the DataFrame for table_names[i].
    If kid_id is None, uses the first kid in that family.
    root_dir: data directory. Supports both live (kids.json) and backup (family_manifest.json) formats.
    """
    if root_dir is None:
        cwd = os.getcwd()
        candidates = [
            cwd,
            os.path.join(cwd, "backend", "data"),
            os.path.join(cwd, "data"),
            os.path.dirname(cwd),
            os.path.join(os.path.dirname(cwd), "backend", "data"),
        ]
        root_dir = next(
            (
                p
                for p in candidates
                if os.path.exists(os.path.join(p, "kids.json"))
                or os.path.exists(os.path.join(p, "family_manifest.json"))
            ),
            None,
        )
        if root_dir is None:
            raise FileNotFoundError(
                "Could not auto-detect data root. Pass root_dir explicitly."
            )

    # Try kids.json (live data) first, then family_manifest.json (backup)
    kids_json_path = os.path.join(root_dir, "kids.json")
    manifest_path = os.path.join(root_dir, "family_manifest.json")

    if os.path.exists(kids_json_path):
        with open(kids_json_path, "r") as f:
            metadata = json.load(f)
        all_kids = metadata.get("kids", [])
    elif os.path.exists(manifest_path):
        with open(manifest_path, "r") as f:
            metadata = json.load(f)
        all_kids = metadata.get("kids", [])
        family_id = metadata.get("family", {}).get("id", family_id)
    else:
        raise FileNotFoundError(f"No kids.json or family_manifest.json found in {root_dir}")
    
    for k in all_kids:
        print(f"{k['id']}={k['name']}, familyID={k['familyId']}")
    
    if kid_id is None:
        family_kids = [k for k in all_kids if str(k["familyId"]) == str(family_id)]
        if not family_kids:
            raise ValueError(f"No kids found for family {family_id}")
        kid_id = family_kids[0]["id"]
    
    print(f"load kid={kid_id}")

    db_path = os.path.join(root_dir, "families", f"family_{family_id}", f"kid_{kid_id}.db")
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Kid DB not found: {db_path}")
    conn = duckdb.connect(db_path, read_only=True)

    tables = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]
    dfs = [conn.execute(f"SELECT * FROM {t}").fetchdf() for t in tables]

    conn.close()
    return tables, dfs

## download

In [56]:
ROOT = "/Users/chen/Library/Mobile Documents/com~apple~CloudDocs/Downloads/"
import glob
RR = None
DD = []
for x in glob.glob(f"{ROOT}/*"):
    if x.endswith("zip"):
        continue
    RR = x
    print(RR)
    
    # From a backup/download
    def get_tables(RR):
        tables, dfs = load_all_tables(family_id=1, root_dir=RR, kid_id=2)
        for i, name in enumerate(tables):
            print(f"dfs[{i}] = {name} ({len(dfs[i])} rows)")
        return tables, dfs
    
    _, dfs = get_tables(RR)
    DD.append(dfs)

In [57]:
x = DD[0][0]
x[x.front.str.startswith("第")]
x[x.deck_id == 11]
y = DD[0][3]
DD[0][2]
DD[0][4]
y[y.session_id == 17]

IndexError: list index out of range

In [ ]:
dfs[0].head(3)

,id,deck_id,front,back,skip_practice,hardness_score,created_at
0,11,2,好像,好像,False,0.0,2026-02-16 06:30:57.851
1,12,2,香,香,False,0.0,2026-02-16 06:31:35.498
2,13,2,菜,菜,False,0.0,2026-02-16 06:31:58.316


In [ ]:
dfs[5]

,id,type,deck_id,planned_count,started_at,completed_at
0,1,flashcard,1,80,2026-02-15 23:21:27.003,2026-02-15 23:25:32.545000
1,6,math,<NA>,20,2026-02-16 15:21:20.841,2026-02-16 15:22:49.440000
2,7,flashcard,1,80,2026-02-16 15:23:27.758,2026-02-16 15:26:24.997000
3,8,math,<NA>,40,2026-02-16 19:36:24.469,2026-02-16 19:40:32.420000
4,10,writing,2,19,2026-02-16 20:32:52.525,2026-02-16 20:40:23.051000
5,12,math,<NA>,40,2026-02-17 22:31:37.224,2026-02-17 22:34:51.176000
6,13,flashcard,1,80,2026-02-17 22:35:21.823,2026-02-17 22:38:16.399000
7,14,flashcard,1,80,2026-02-18 22:17:42.158,2026-02-18 22:26:02.672411
8,15,math,<NA>,40,2026-02-18 22:26:20.776,2026-02-18 22:31:22.225809
9,16,math,<NA>,40,2026-02-19 23:10:32.802,2026-02-19 23:17:48.899884


# from local

In [39]:
# From local
tables, dfs = load_all_tables(family_id=1, kid_id=1)
for i, name in enumerate(tables):
    print(f"dfs[{i}] = {name} ({len(dfs[i])} rows)")

1=Maria, familyID=1
2=Alex, familyID=1
3=fred, familyID=1
load kid=1
dfs[0] = cards (586 rows)
dfs[1] = deck_category_opt_in (7 rows)
dfs[2] = decks (8 rows)
dfs[3] = lesson_reading_audio (0 rows)
dfs[4] = session_results (2201 rows)
dfs[5] = sessions (46 rows)
dfs[6] = writing_sheet_cards (22 rows)
dfs[7] = writing_sheets (4 rows)


In [41]:
x = dfs[0]
#y = dfs[8]
#z = x.merge(y, left_on="id", right_on="card_id", how="inner")
#z
dfs[5]

,id,type,planned_count,started_at,completed_at,retry_count,retry_total_response_ms,retry_best_rety_correct_count
0,1,chinese_characters,80,2026-02-15 23:21:27.003,2026-02-15 23:25:32.545000,0,0,75
1,7,chinese_characters,80,2026-02-16 15:23:27.758,2026-02-16 15:26:24.997000,0,0,80
2,13,chinese_characters,80,2026-02-17 22:35:21.823,2026-02-17 22:38:16.399000,0,0,80
3,14,chinese_characters,80,2026-02-18 22:17:42.158,2026-02-18 22:26:02.672411,0,0,73
4,17,chinese_characters,80,2026-02-19 23:35:12.361,2026-02-19 23:40:51.150223,0,0,73
5,18,chinese_characters,80,2026-02-20 23:40:16.429,2026-02-20 23:43:39.177843,0,0,74
6,21,chinese_characters,80,2026-02-21 15:09:49.916,2026-02-21 15:14:51.337558,0,0,74
7,25,chinese_characters,80,2026-02-22 18:14:25.000,2026-02-22 18:20:01.353985,0,0,76
8,28,chinese_characters,80,2026-02-23 18:16:51.120,2026-02-23 18:29:14.291297,0,0,76
9,29,chinese_characters,80,2026-02-24 20:16:42.622,2026-02-24 20:23:09.831840,0,0,68


In [10]:
x = dfs[3]
x[x.session_id == 13]
y = dfs[0]
y[y.id.isin([511, 512, 513])]

,id,deck_id,front,back,skip_practice,hardness_score,created_at
250,511,13,公鸡蛋（上）,50,False,0.0,2026-02-17 23:13:23.716
251,512,13,女娲造人,54,False,0.0,2026-02-17 23:13:23.728
252,513,13,叶公好龙,55,False,0.0,2026-02-17 23:13:23.737


In [295]:
dfs[4]

,id,type,deck_id,planned_count,started_at,completed_at
0,9,writing,2,15,2026-02-17 21:15:28.004,2026-02-17 21:22:31.124000
1,10,flashcard,1,40,2026-02-18 21:53:42.428,2026-02-18 21:55:19.559494
2,11,flashcard,1,40,2026-02-19 13:48:19.304,2026-02-19 13:50:23.471676
3,12,writing,2,15,2026-02-19 21:14:02.364,2026-02-19 21:20:35.567649
4,13,lesson_reading,<NA>,3,2026-02-20 12:26:25.071,2026-02-20 12:38:28.165714
5,14,flashcard,1,40,2026-02-20 12:38:45.019,2026-02-20 12:40:34.189013
6,15,writing,2,15,2026-02-20 12:41:05.903,2026-02-20 12:45:45.993564
7,16,flashcard,1,40,2026-02-21 18:09:04.383,2026-02-21 18:10:41.446459
8,17,lesson_reading,<NA>,3,2026-02-21 18:10:57.454,2026-02-21 18:44:03.003375
9,18,writing,2,15,2026-02-21 18:49:24.423,2026-02-21 18:53:47.056094


In [172]:
dfs[1].tail()

,id,name,description,tags,created_at,updated_at
5,6,Math Subtraction Within 20,All ordered pairs where a - b and a is between...,"[math, subtraction, within20]",2026-02-16 19:32:50.004,2026-02-16 19:32:50.004
6,7,Chinese Reading Ma3 Unit 1,马三 第一单元 课文读诵,"[lesson_reading, maliping, ma3, unit1]",2026-02-17 23:13:23.410,2026-02-17 23:13:23.410
7,8,Chinese Reading Ma3 Unit 2,马三 第二单元 课文读诵,"[lesson_reading, maliping, ma3, unit2]",2026-02-17 23:13:23.432,2026-02-17 23:13:23.432
8,9,Chinese Reading Ma3 Unit 3,马三 第三单元 课文读诵,"[lesson_reading, maliping, ma3, unit3]",2026-02-17 23:13:23.453,2026-02-17 23:13:23.453
9,13,math_orphan,Reserved deck for orphaned math cards,"[math, orphan]",2026-02-20 01:48:16.807,2026-02-20 01:48:16.807


In [296]:
dfs[4]

,id,type,deck_id,planned_count,started_at,completed_at
0,9,writing,2,15,2026-02-17 21:15:28.004,2026-02-17 21:22:31.124000
1,10,flashcard,1,40,2026-02-18 21:53:42.428,2026-02-18 21:55:19.559494
2,11,flashcard,1,40,2026-02-19 13:48:19.304,2026-02-19 13:50:23.471676
3,12,writing,2,15,2026-02-19 21:14:02.364,2026-02-19 21:20:35.567649
4,13,lesson_reading,<NA>,3,2026-02-20 12:26:25.071,2026-02-20 12:38:28.165714
5,14,flashcard,1,40,2026-02-20 12:38:45.019,2026-02-20 12:40:34.189013
6,15,writing,2,15,2026-02-20 12:41:05.903,2026-02-20 12:45:45.993564
7,16,flashcard,1,40,2026-02-21 18:09:04.383,2026-02-21 18:10:41.446459
8,17,lesson_reading,<NA>,3,2026-02-21 18:10:57.454,2026-02-21 18:44:03.003375
9,18,writing,2,15,2026-02-21 18:49:24.423,2026-02-21 18:53:47.056094


In [23]:
# Load shared personal deck DB (global across all families)
from pathlib import Path
import duckdb

candidate_paths = [
    Path('shared_decks.duckdb'),
    Path('data/shared_decks.duckdb'),
    Path('backend/data/shared_decks.duckdb'),
    Path('../data/shared_decks.duckdb'),
]
shared_db_path = next((p.resolve() for p in candidate_paths if p.exists()), None)
if shared_db_path is None:
    raise FileNotFoundError('Could not find shared_decks.duckdb from current working directory')

conn = duckdb.connect(str(shared_db_path), read_only=True)
shared_tables = [row[0] for row in conn.execute('SHOW TABLES').fetchall()]
shared_dfs = {table: conn.execute(f'SELECT * FROM {table}').fetchdf() for table in shared_tables}
conn.close()

print(f'Loaded shared DB: {shared_db_path}')
for table in shared_tables:
    print(f'shared_dfs[\"{table}\"]: {len(shared_dfs[table])} rows')
    
shared_tables


Loaded shared DB: /Users/chen/Library/Mobile Documents/com~apple~CloudDocs/Project/backend/data/shared_decks.duckdb
shared_dfs["cards"]: 2665 rows
shared_dfs["deck"]: 277 rows
shared_dfs["deck_category"]: 7 rows


['cards', 'deck', 'deck_category']

In [14]:
shared_dfs['deck_category']

,category_key,behavior_type,has_chinese_specific_logic,display_name,emoji,is_shared_with_non_super_family
0,math,type_i,False,Math,➗,True
1,chinese_characters,type_i,True,Chinese Characters,📖,True
2,chinese_writing,type_ii,True,Chinese Writing,✍️,True
3,chinese_reading,type_iii,True,Chinese Reading,📚,True


In [26]:
shared_dfs['cards'].tail(20)

,id,deck_id,front,back
2645,2780,279,9 × 6,54
2646,2781,279,7 × 7,49
2647,2782,279,7 × 8,56
2648,2783,279,8 × 7,56
2649,2784,279,7 × 9,63
2650,2785,279,9 × 7,63
2651,2786,279,8 × 8,64
2652,2787,279,8 × 9,72
2653,2788,279,9 × 8,72
2654,2789,279,9 × 9,81


In [25]:
shared_dfs['deck']

,deck_id,name,tags,creator_family_id,created_at
0,1,math_add_within10,"[math, add, within10]",1,2026-02-20 22:50:03.386358
1,2,math_sub_within10,"[math, sub, within10]",1,2026-02-20 22:52:21.756814
2,3,chinese_reading_ma3_unit1_week1,"[chinese_reading, ma3(马立平3年级), unit1, week1]",1,2026-02-22 00:44:11.252923
3,4,chinese_reading_ma3_unit1_week2,"[chinese_reading, ma3(马立平3年级), unit1, week2]",1,2026-02-22 00:44:58.686635
4,6,chinese_reading_ma3_unit1_week3,"[chinese_reading, ma3(马立平3年级), unit1, week3]",1,2026-02-22 00:47:53.224089
...,...,...,...,...,...
272,279,math_multi_9x9,"[math, multi, 9x9]",1,2026-03-01 02:00:23.429104
273,282,musical_instruments_strings,"[musical_instruments, strings]",1,2026-03-04 21:06:38.939148
274,283,musical_instruments_brass,"[musical_instruments, brass]",1,2026-03-04 21:07:19.341694
275,284,piano_book4,"[piano, book4]",1,2026-03-04 22:29:51.577417
